In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import glob
import os
images_path = glob.glob(os.path.join(path, "dataset/images", "*"))
masks_path = glob.glob(os.path.join(path, "dataset/masks", "*"))

images_path.sort()
masks_path.sort()
images_path[0], masks_path[0]

In [ ]:

# TO DO

import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class FloodSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
      # TODO: Load the image and mask at index idx
      # Hint: Use Image.open() and convert image to "RGB", mask to "L" (grayscale)
      image_path = self.image_paths[idx]
      mask_path = self.mask_paths[idx]
      # YOUR CODE HERE
      image = Image.open(image_path)
      image = image.convert("RGB")
      mask = Image.open(mask_path)
      mask = mask.convert("L")

      # Apply transforms
      if self.transform:
        image = self.transform(image)

      if self.target_transform:
        mask = self.target_transform(mask)
        mask = remap_mask(mask)  # Convert to binary mask

      return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
  # TODO: Add transforms
  # Hint: ToTensor, Resize to (256, 256), Normalize with ImageNet mean/std
  # YOUR CODE HERE
  transforms.ToTensor(),
  transforms.Resize((256, 256)),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mask transforms (Resize, PILToTensor)
mask_transforms = transforms.Compose([
  # TODO: Add transforms
  # Hint: Resize to (256, 256) with NEAREST interpolation, PILToTensor
  # YOUR CODE HERE
  transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])

# Split into train and test sets (80% train, 20% test)

# YOUR CODE HERE
train_images, test_images, train_masks, test_masks = train_test_split(
  images_path, masks_path, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = FloodSegmentationDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = FloodSegmentationDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img

In [ ]:
# Display 4 images with their masks side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # TODO: Get an image-mask pair from train_dataset
    # Hint: Use train_dataset[i] to get the i-th sample

    # YOUR CODE HERE
    img, mask = train_dataset[i]

    # Display image (denormalize first)
    axes[0, i].imshow(denormalize(img))
    axes[0, i].set_title(f"MRI Image {i+1}")
    axes[0, i].axis("off")

    # Display mask
    axes[1, i].imshow(mask.squeeze(), cmap="gray")
    axes[1, i].set_title(f"Tumor Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
# TO DO

import segmentation_models_pytorch as smp

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
model

In [ ]:
# TO DO

import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        # For CrossEntropyLoss, masks should be type long and without channel dim (C=1)
        images, masks = images.to(device), masks.to(device).squeeze(1).long()

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            # For CrossEntropyLoss, masks should be type long and without channel dim (C=1)
            images, masks = images.to(device), masks.to(device).squeeze(1).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# TO DO

import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss() # Changed to CrossEntropyLoss for multi-class segmentation
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 5 # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TODO: Task 8 - Visualize predictions on test images
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  # TODO: Get model prediction
  # 1. Add batch dimension: image.unsqueeze(0)
  # 2. Move to device
  # 3. Get prediction: model(image)
  # 4. Apply sigmoid to get probabilities
  # 5. Threshold at 0.5 to get binary mask

  with torch.no_grad():
    pred_mask_logits = model(img.unsqueeze(0).to(device))  # Forward pass
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)
    pred_mask = torch.argmax(pred_mask_logits, dim=1).squeeze(0).cpu().numpy()
    # i edited here the segmentation solution and change from binary

  # Display results
  axes[i, 0].imshow(denormalize(image))
  axes[i, 0].set_title("MRI Image")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred_mask.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()